# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 7.8 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict
from onnx import shape_inference

In [5]:
TASK_ID = "task224"
CH = 10
H = W = 30
LOCAL_TASK_JSON = Path("/mnt/data/task224(1).json")
KAGGLE_TASK_JSON = Path(COMPETITION) / "task224.json"
TASK_JSON = LOCAL_TASK_JSON if LOCAL_TASK_JSON.exists() else KAGGLE_TASK_JSON
OUT_DIR = Path.cwd() / "task224_anchor_rectangle_canvasmask_30x30"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUBMISSION_PATH = Path.cwd() / "submission.zip"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_anchor_rectangle_canvasmask_validation_summary.json"

with TASK_JSON.open("r") as f:
    task = json.load(f)

len(task["train"]), len(task["test"]), len(task["arc-gen"])


(3, 1, 262)

In [6]:
def grid_to_tensor_zero_padded(grid, h=H, w=W, ch=CH):
    """Convert ARC grid to [1,10,30,30].
    Inside the real grid: one-hot, including background color 0.
    Outside the real grid: all-zero across every channel.
    """
    x = np.zeros((1, ch, h, w), dtype=np.float32)
    for r, row in enumerate(grid):
        for c, v in enumerate(row):
            x[0, int(v), r, c] = 1.0
    return x

def python_rule(grid):
    arr = np.array(grid, dtype=np.int64)
    out = arr.copy()
    marker = np.argwhere(arr == 5)
    rmin, cmin = marker.min(axis=0)
    rmax, cmax = marker.max(axis=0)
    top, bottom = int(rmin) + 1, int(rmax) - 1
    left, right = int(cmin) + 1, int(cmax) - 1
    object_colors = [int(v) for v in np.unique(arr) if v not in (0, 5)]
    color = object_colors[0]
    out[top, left:right + 1] = color
    out[bottom, left:right + 1] = color
    out[top:bottom + 1, left] = color
    out[top:bottom + 1, right] = color
    out[arr == 5] = 5
    return out.tolist()

for split in ["train", "test", "arc-gen"]:
    ok = sum(python_rule(ex["input"]) == ex["output"] for ex in task[split])
    print(split, ok, "/", len(task[split]))


train 3 / 3
test 1 / 1
arc-gen 262 / 262


In [7]:
class Task224Model(nn.Module):
    def __init__(self, h=H, w=W):
        super().__init__()
        rr = torch.arange(h, dtype=torch.float32).view(1, 1, h, 1).expand(1, 1, h, w)
        cc = torch.arange(w, dtype=torch.float32).view(1, 1, 1, w).expand(1, 1, h, w)
        self.register_buffer("R", rr)
        self.register_buffer("C", cc)

    def forward(self, x):
        active = (x.sum(dim=1, keepdim=True) > 0.5).float()
        m5 = x[:, 5:6, :, :]
        big = torch.tensor(1000.0, dtype=torch.float32, device=x.device)
        rmin = torch.amin(m5 * self.R + (1.0 - m5) * big, dim=(2, 3), keepdim=True)
        rmax = torch.amax(m5 * self.R - (1.0 - m5) * big, dim=(2, 3), keepdim=True)
        cmin = torch.amin(m5 * self.C + (1.0 - m5) * big, dim=(2, 3), keepdim=True)
        cmax = torch.amax(m5 * self.C - (1.0 - m5) * big, dim=(2, 3), keepdim=True)

        top, bottom = rmin + 1.0, rmax - 1.0
        left, right = cmin + 1.0, cmax - 1.0
        on_top = (self.R - top).abs() < 0.25
        on_bottom = (self.R - bottom).abs() < 0.25
        on_left = (self.C - left).abs() < 0.25
        on_right = (self.C - right).abs() < 0.25
        within_cols = (self.C >= left) & (self.C <= right)
        within_rows = (self.R >= top) & (self.R <= bottom)
        border = ((((on_top | on_bottom) & within_cols) | ((on_left | on_right) & within_rows)).float()) * active

        colored = []
        for k in range(1, 10):
            xk = x[:, k:k + 1, :, :]
            if k == 5:
                ch = xk * active
            else:
                present = (xk.sum(dim=(2, 3), keepdim=True) > 0.5).float()
                ch = ((xk + present * border) > 0.5).float() * active
            colored.append(ch)
        occupied = (torch.cat(colored, dim=1).sum(dim=1, keepdim=True) > 0.5).float()
        ch0 = active * (1.0 - occupied)
        return torch.cat([ch0] + colored, dim=1)

model = Task224Model().eval()


In [8]:
dummy = torch.from_numpy(grid_to_tensor_zero_padded(task["test"][0]["input"]))

torch.onnx.export(
    model,
    dummy,
    str(ONNX_PATH),
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
)

# Save shape-inferred model so intermediate value_info tensors are statically described.
onnx_model = onnx.load(str(ONNX_PATH))
onnx_model = onnx.shape_inference.infer_shapes(onnx_model)
onnx.save(onnx_model, str(ONNX_PATH))
onnx.checker.check_model(str(ONNX_PATH))

ONNX_PATH, ONNX_PATH.stat().st_size


/tmp/ipykernel_16/2362384443.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(
/tmp/ipykernel_16/1151077928.py:12: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  big = torch.tensor(1000.0, dtype=torch.float32, device=x.device)


(PosixPath('/kaggle/working/task224_anchor_rectangle_canvasmask_30x30/task224.onnx'),
 30372)

In [9]:
def vi_shape(vi):
    dims = []
    for d in vi.type.tensor_type.shape.dim:
        if d.dim_value:
            dims.append(int(d.dim_value))
        elif d.dim_param:
            dims.append(str(d.dim_param))
        else:
            dims.append(None)
    return dims

onnx_model = onnx.load(str(ONNX_PATH))
ops = collections.Counter(node.op_type for node in onnx_model.graph.node)
forbidden = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
empty_inputs = [
    (node.name, node.op_type, list(node.input))
    for node in onnx_model.graph.node
    if any(inp == "" for inp in node.input)
]
bad_shapes = []
for vi in list(onnx_model.graph.input) + list(onnx_model.graph.value_info) + list(onnx_model.graph.output):
    shp = vi_shape(vi)
    if any(d is None or isinstance(d, str) for d in shp):
        bad_shapes.append((vi.name, shp))

print("input shape:", vi_shape(onnx_model.graph.input[0]))
print("output shape:", vi_shape(onnx_model.graph.output[0]))
print("ONNX size:", ONNX_PATH.stat().st_size)
print("ops:", dict(ops))
print("forbidden ops:", sorted(forbidden & set(ops)))
print("empty optional inputs:", len(empty_inputs))
print("non-static tensor shapes:", len(bad_shapes))

assert vi_shape(onnx_model.graph.input[0]) == [1, 10, 30, 30]
assert vi_shape(onnx_model.graph.output[0]) == [1, 10, 30, 30]
assert not (forbidden & set(ops))
assert not empty_inputs
assert not bad_shapes
assert ONNX_PATH.stat().st_size < 1_440_000


input shape: [1, 10, 30, 30]
output shape: [1, 10, 30, 30]
ONNX size: 30372
ops: {'Constant': 67, 'ReduceSum': 10, 'Greater': 18, 'Cast': 19, 'Slice': 9, 'Mul': 22, 'Sub': 10, 'Add': 12, 'ReduceMin': 2, 'ReduceMax': 2, 'Abs': 4, 'Less': 4, 'GreaterOrEqual': 2, 'LessOrEqual': 2, 'And': 4, 'Or': 3, 'Concat': 2}
forbidden ops: []
empty optional inputs: 0
non-static tensor shapes: 0


In [10]:
sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1
sess = ort.InferenceSession(str(ONNX_PATH), sess_options=sess_options, providers=["CPUExecutionProvider"])

def validate_examples(examples):
    tensor_ok = 0
    grid_ok = 0
    outside_zero_ok = 0
    bad = []
    for i, ex in enumerate(examples):
        x = grid_to_tensor_zero_padded(ex["input"])
        y = sess.run(None, {"input": x})[0]
        exp = grid_to_tensor_zero_padded(ex["output"])
        pred_bin = (y > 0.5).astype(np.float32)

        if np.array_equal(pred_bin, exp):
            tensor_ok += 1
        else:
            bad.append(i)

        h, w = len(ex["output"]), len(ex["output"][0])
        pred_grid = pred_bin[0, :, :h, :w].argmax(axis=0).astype(np.int64).tolist()
        if pred_grid == ex["output"]:
            grid_ok += 1

        active = x.sum(axis=1, keepdims=True) > 0.5
        if np.all(np.abs(y * (~active)) < 1e-5):
            outside_zero_ok += 1

    return {
        "tensor_exact_zero_padded": [tensor_ok, len(examples)],
        "grid_argmax_inside_canvas": [grid_ok, len(examples)],
        "outside_active_all_channels_zero": [outside_zero_ok, len(examples)],
        "bad_indices": bad[:10],
    }

def validate_split(split):
    return validate_examples(task[split])

rng = random.Random(0)
arcgen_indices = list(range(len(task["arc-gen"])))
rng.shuffle(arcgen_indices)
holdout_n = max(1, int(math.ceil(0.60 * len(arcgen_indices))))
arcgen_holdout = [task["arc-gen"][i] for i in arcgen_indices[:holdout_n]]

summary = {
    "task_id": TASK_ID,
    "rule": "draw the object-colored outer hollow rectangle one step inside the four marker-5 extrema, while preserving markers and the inner object",
    "onnx_path": str(ONNX_PATH),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "input_shape": vi_shape(onnx_model.graph.input[0]),
    "output_shape": vi_shape(onnx_model.graph.output[0]),
    "ops": dict(ops),
    "forbidden_ops": sorted(forbidden & set(ops)),
    "empty_optional_inputs": len(empty_inputs),
    "non_static_tensor_shapes": len(bad_shapes),
    "arc_gen_holdout_policy": "deterministic random seed 0, 60% of arc-gen; full arc-gen also validated",
    "validation": {
        "train": validate_split("train"),
        "test": validate_split("test"),
        "arc-gen_60pct_holdout": validate_examples(arcgen_holdout),
        "arc-gen_full": validate_split("arc-gen"),
    },
}

print(json.dumps(summary, indent=2)[:6000])
with SUMMARY_PATH.open("w") as f:
    json.dump(summary, f, indent=2)

assert summary["validation"]["train"]["tensor_exact_zero_padded"][0] == summary["validation"]["train"]["tensor_exact_zero_padded"][1]
assert summary["validation"]["test"]["tensor_exact_zero_padded"][0] == summary["validation"]["test"]["tensor_exact_zero_padded"][1]
assert summary["validation"]["arc-gen_60pct_holdout"]["tensor_exact_zero_padded"][0] == summary["validation"]["arc-gen_60pct_holdout"]["tensor_exact_zero_padded"][1]
assert summary["validation"]["arc-gen_full"]["tensor_exact_zero_padded"][0] == summary["validation"]["arc-gen_full"]["tensor_exact_zero_padded"][1]


{
  "task_id": "task224",
  "rule": "draw the object-colored outer hollow rectangle one step inside the four marker-5 extrema, while preserving markers and the inner object",
  "onnx_path": "/kaggle/working/task224_anchor_rectangle_canvasmask_30x30/task224.onnx",
  "onnx_size_bytes": 30372,
  "input_shape": [
    1,
    10,
    30,
    30
  ],
  "output_shape": [
    1,
    10,
    30,
    30
  ],
  "ops": {
    "Constant": 67,
    "ReduceSum": 10,
    "Greater": 18,
    "Cast": 19,
    "Slice": 9,
    "Mul": 22,
    "Sub": 10,
    "Add": 12,
    "ReduceMin": 2,
    "ReduceMax": 2,
    "Abs": 4,
    "Less": 4,
    "GreaterOrEqual": 2,
    "LessOrEqual": 2,
    "And": 4,
    "Or": 3,
    "Concat": 2
  },
  "forbidden_ops": [],
  "empty_optional_inputs": 0,
  "non_static_tensor_shapes": 0,
  "arc_gen_holdout_policy": "deterministic random seed 0, 60% of arc-gen; full arc-gen also validated",
  "validation": {
    "train": {
      "tensor_exact_zero_padded": [
        3,
        3
      ]

In [11]:
with zipfile.ZipFile(SUBMISSION_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("Wrote:", SUBMISSION_PATH)
print("Zip contents:", zipfile.ZipFile(SUBMISSION_PATH).namelist())
assert zipfile.ZipFile(SUBMISSION_PATH).namelist() == [f"{TASK_ID}.onnx"]


Wrote: /kaggle/working/submission.zip
Zip contents: ['task224.onnx']
